## Two-Qubit Tutorial

In [1]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from qsopt.core.experimental_parameters import (
    ExperimentalParameters,
    PhysicalConstants,
    SystemDimensions,
    MeasurementProtocol,
    InteractionType,
    QubitInteraction,
    InitialStateConfig,
    InitialStateType,
    NoiseConfiguration
)
from qsopt.core.trainable_parameters import TrainableParameters
from qsopt.core.experiment.two_qubit_experiment import TwoQubitExperiment

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

## 1. Setup Experimental Parameters

In [2]:
# Define experimental parameters using nested configuration objects
interaction = QubitInteraction(
    qubit_indices=(0, 1),
    interaction_type=InteractionType.XX,
    chi=0.1
)

physical_constants = PhysicalConstants(
    n_qubits=2,
    chi=[30.0, 30.0],
    photon_cavity_coupling=15.0,
    inverse_pulse_width=1.0,
    qubit_interactions=[interaction]
)

system_dims = SystemDimensions(
    field_levels=2,
    cavity_levels=2,
    qubit_levels=[2, 2]
)

measurement = MeasurementProtocol(
    measurement_times=[-8.0, 8.0] 
)

initial_state = InitialStateConfig(state_type=InitialStateType.SINGLE_PHOTON)

noise_config = NoiseConfiguration(
    depolarizing=[0.0, 0.0],
    dephasing=[0.0, 0.0],
    relaxation=[0.0, 0.0]
)

exp_params = ExperimentalParameters(
    physical_constants=physical_constants,
    system_dims=system_dims,
    measurement=measurement,
    initial_state=initial_state,
    noise_config=noise_config
)

print(exp_params)

SYSTEM DIMENSIONS
  Number of qubits:          2
  Cavity levels:             2
  Qubit levels:         [2, 2]
  Field levels:              2
  Total dimension:          16
PHYSICAL CONSTANTS
  Chi:                  [30.0, 30.0]
  Photon cavity coupling: 15.0000
  Inverse pulse width:    1.0000
  Qubit interactions:   1 interaction(s)
    [0] Qubits (0, 1): sx-sx, χ=0.1000
MEASUREMENT PROTOCOL
  Mode:                 Explicit list
  Number of measurements:      2
  Measurement times:    [-8.0, 8.0]
INITIAL STATE
  Type:                 single_photon
NOISE MODEL
  Depolarizing rate:    [0.0, 0.0]
  Dephasing rate:       [0.0, 0.0]
  Relaxation rate:      [0.0, 0.0]
  Custom operators:     None
SYSTEM STATUS
  Configuration:        VALID


## 2. Setup Trainable Parameters

For two-qubit experiments, we need 4 rotation angles:
- θ1_q1, θ2_q1: Rotation angles for qubit 1
- θ1_q2, θ2_q2: Rotation angles for qubit 2

In [10]:
# Create trainable parameters with 4 rotation angles
trainable_params = TrainableParameters()

import optax
optimizer = optax.sgd(learning_rate=1.)

trainable_params.add_rotation_angles(
    names=["theta1_q1", "theta2_q1", "theta1_q2", "theta2_q2"],
    initial_values=[np.pi/2, -np.pi/2, np.pi/2, -np.pi/2],

)

print(trainable_params)

Trainable Parameters: 4
  Rotation Angles:
    theta1_q1: 1.5708 rad (90.00°)
    theta2_q1: -1.5708 rad (-90.00°)
    theta1_q2: 1.5708 rad (90.00°)
    theta2_q2: -1.5708 rad (-90.00°)


## 3. Create Two-Qubit Experiment

Initialize the experiment with our parameters. This will:
- Generate operators for the 4-subsystem composite space
- Build the time-dependent Hamiltonian
- Cache initial state and projectors

In [11]:
experiment = TwoQubitExperiment(exp_params, trainable_params)

## 4. Run Basic Simulation

In [12]:
# Run simulation with zero rotations
callback = experiment.run_simulation(batch_size=1)

print(callback)

MODE: Single Simulation
  Current Parameters:
     theta1_q1: 1.570796 rad (90.00°)
     theta2_q1: -1.570796 rad (-90.00°)
     theta1_q2: 1.570796 rad (90.00°)
     theta2_q2: -1.570796 rad (-90.00°)
  Detection Probabilities:
     P(with photon):    0.989767
     P(without photon): 0.000000
     Contrast:          0.989767


## 5. Optimize Rotation Angles

Now let's optimize the four rotation angles to maximize sensing contrast. By default, the optimization maximizes the detection probability `1 - P(00)` (any outcome except ground state).


In [13]:
# Optimize with default detection criterion (1 - P(00))
callback_opt = experiment.optimize_rotations(
    num_steps=500,
    batch_size=1,
    tolerance=1e-8,
    verbose=True,
    verbose_step=50,
    theta_init=[np.pi/3, -np.pi/3, np.pi/4, -np.pi]
)


Configuration:
    Max iterations: 500
    Batch size: 1
    Convergence tolerance: 1.00e-08
    Detection criterion: 1 - P(00)
    Initial rotation parameters:
        theta1_q1=1.047 rad
        theta2_q1=-1.047 rad
        theta1_q2=0.785 rad
        theta2_q2=-3.142 rad
Step  theta1_q1   theta2_q1   theta1_q2   theta2_q2   Contrast    Grad Norm
----------------------------------------------------------------------
0     1.047198    -1.047198   0.785398    -3.141593   0.118882    5.15e-01    
50    1.040621    -1.075213   0.965112    -2.909738   0.294140    6.60e-01    
100   1.047375    -1.090238   1.185777    -2.643821   0.533769    6.99e-01    
150   1.062124    -1.097978   1.396310    -2.392822   0.749141    5.91e-01    


KeyboardInterrupt: 

### Visualize Optimization Progress

Let's plot how the contrast evolved during optimization:


In [14]:
from qsopt.utils.visualization import plot_optimization_dashboard

# Create comprehensive dashboard showing all optimization metrics
fig = plot_optimization_dashboard(
    callback_opt,
    show_contrast=True,
    show_gradients=True,
    show_parameters=True,
    show_trajectory=True,
    show_probabilities=True,
    figsize=(16, 14)
)
plt.show()


NameError: name 'callback_opt' is not defined